# Gold dimension: `dim_date`

## Purpose

The shared calendar for every Gold date key. `fact_project_snapshot` joins to it three times: snapshot, start, and source completion date.

## Grain

One row per calendar date from 2000-01-01 to 2035-12-31, plus key `0`. Uniqueness: `calendar_date`.

## Keys

Primary key `date_key`. Natural key `calendar_date`. Recipe: the date as `INT` in `yyyymmdd` form, such as `20261004` (D-33).

## Key 0 member

`date_key = 0`, a `NULL` date, and `month_name = 'Unknown'`. Facts use it when a source date is missing.

## Inputs

None. `SEQUENCE` generates the dates.

## Outputs

`03-gold.dim_date`, loaded with `MERGE` on `date_key`. Rows are never deleted.

## Not done here

Range coverage of Silver dates is checked by the Gold validator, not here.

## 1. Declare the calendar range

**What:** declare the first and last calendar date once.

**Why:** the range must cover every project start date, completion date, and snapshot date. Silver project dates begin in 2003, and the current snapshot is from 2026. The range runs from 2000 to 2035, so new snapshots fit without a change.

**Protects:** the Gold validator stops the run if any fact date falls outside this range.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `03-gold`;

-- covers every silver project date (2003 to 2026) with room for future snapshots
DECLARE OR REPLACE VARIABLE calendar_start_date DATE DEFAULT DATE '2000-01-01';
-- last calendar date. new snapshots up to 2035 get a date key without a change (d-33)
DECLARE OR REPLACE VARIABLE calendar_end_date DATE DEFAULT DATE '2035-12-31';

## 2. Create the table

**What:** create `03-gold.dim_date` with the column names, types, and order in `docs/gold_model.md`.

**Why:** Genie reads the table and column comments, so every column has one.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `03-gold`.dim_date (
    date_key INT NOT NULL COMMENT 'Surrogate key in yyyymmdd form, such as 20261004. Key 0 is the unknown date.',
    calendar_date DATE COMMENT 'Calendar date. NULL only for the unknown member, key 0.',
    calendar_year INT COMMENT 'Four-digit calendar year.',
    calendar_quarter INT COMMENT 'Calendar quarter from 1 to 4.',
    month_number INT COMMENT 'Calendar month from 1 (January) to 12 (December).',
    month_name STRING COMMENT 'Full English month name, such as October. Unknown for key 0.',
    day_of_month INT COMMENT 'Day of the month from 1 to 31.'
)
USING DELTA
COMMENT 'Gold calendar dimension. One row per calendar date plus key 0 for a missing date. Facts join to it once per date role: snapshot, start, and source completion.';

## 3. Build and merge the calendar rows

**What:** generate one row per date in the range, add the unknown member, and merge them by `date_key`.

**Why:** `SEQUENCE` builds the dates without reading any table. The key is derived from the date itself, so it never changes between runs.

**Expected validation:** keys are unique, and exactly one row has key `0`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW dim_date_source AS
WITH calendar AS (
    SELECT EXPLODE(SEQUENCE(calendar_start_date, calendar_end_date, INTERVAL 1 DAY)) AS calendar_date
),
known_dates AS (
    SELECT
        CAST(DATE_FORMAT(calendar.calendar_date, 'yyyyMMdd') AS INT) AS date_key,
        calendar.calendar_date,
        YEAR(calendar.calendar_date) AS calendar_year,
        QUARTER(calendar.calendar_date) AS calendar_quarter,
        MONTH(calendar.calendar_date) AS month_number,
        DATE_FORMAT(calendar.calendar_date, 'MMMM') AS month_name,
        DAYOFMONTH(calendar.calendar_date) AS day_of_month
    FROM calendar
),
unknown_date AS (
    SELECT
        0 AS date_key,
        CAST(NULL AS DATE) AS calendar_date,
        CAST(NULL AS INT) AS calendar_year,
        CAST(NULL AS INT) AS calendar_quarter,
        CAST(NULL AS INT) AS month_number,
        'Unknown' AS month_name,
        CAST(NULL AS INT) AS day_of_month
)
SELECT * FROM unknown_date
UNION ALL
SELECT * FROM known_dates;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT date_key) AND COUNT_IF(date_key = 0) = 1,
    'STOP: dim_date keys are not unique or the unknown member is missing'
)
FROM dim_date_source;

MERGE INTO `03-gold`.dim_date AS target
USING dim_date_source AS source
ON target.date_key = source.date_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

# Summary

Built `03-gold.dim_date` at one row per calendar date from 2000-01-01 to 2035-12-31, plus key `0`. The query below shows the row count and range. Known limit: coverage of every Silver date is checked by the Gold validator, which is not built yet. Next: `01_gold_dim_region`.

In [ ]:
%sql
SELECT
    COUNT(*) AS date_rows,
    COUNT_IF(date_key = 0) AS unknown_rows,
    MIN(calendar_date) AS first_date,
    MAX(calendar_date) AS last_date
FROM `03-gold`.dim_date;